<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Image Processing Fundamental</b></h1>
</div>

This notebook executes the core numerical image-processing experiments covering image representation, pixel and ROI operations, channel conventions, dynamic range, noise models, image-comparison metrics, encoding effects, and validation.


## Setup — Environment and Configuration

We import the libraries used throughout the notebook.

The random generator is seeded so that the synthetic noise examples are reproducible.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Reproducible random generator used for noise experiments.
RNG = np.random.default_rng(42)

# Improve readability of figures without changing global plotting style.
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titlesize"] = 11

### Checkpoint

At this stage, the important libraries are:

- **NumPy** → numerical image arrays;
- **Matplotlib** → visualization;
- **Pillow** → image loading and saving;
- **Pathlib** → robust file paths.

## 1. Data and Output Paths

The notebook is stored under `notebooks/`, while the input images are stored under `data/`.

The path resolver below walks upward from the current working directory until it finds the laboratory root. This makes the notebook robust when VS Code or Jupyter starts from slightly different working directories.

In [ ]:
def find_lab_root(start: Path) -> Path:
    """Locate the lab directory by searching upward for data/ and notebooks/."""
    start = start.resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "data").is_dir() and (candidate / "notebooks").is_dir():
            return candidate

    raise FileNotFoundError(
        "Could not locate the lab root. "
        "Expected a directory containing both 'data/' and 'notebooks/'."
    )


LAB_DIR = find_lab_root(Path.cwd())
DATA_DIR = LAB_DIR / "data"
OUTPUT_DIR = LAB_DIR / "outputs" / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_FILES = {
    "einstein": DATA_DIR / "einstein.png",
    "peppers": DATA_DIR / "peppers.png",
    "ballons": DATA_DIR / "ballons.jpg",
    "grass": DATA_DIR / "grass.jpg",
    "tower": DATA_DIR / "Elizabeth_Tower_London.jpg",
}

missing_files = [path.name for path in IMAGE_FILES.values() if not path.exists()]
assert not missing_files, f"Missing input files: {missing_files}"

print("Lab directory :", LAB_DIR)
print("Data directory:", DATA_DIR)
print("Output folder :", OUTPUT_DIR)
print("Images found  :", len(IMAGE_FILES))

### Interpretation

A reproducible notebook should never depend on a machine-specific path such as `/home/user/...`.

Relative or automatically resolved paths make the same notebook usable on another computer.

## 2. From a Real Scene to a Digital Image

A real-world scene is continuous in space and light intensity. A digital image is not continuous: it is a finite grid of measurements.

A simplified acquisition chain is:

```text
Real scene
    ↓
Optical system
    ↓
Image sensor
    ↓
Spatial sampling
    ↓
Intensity quantization
    ↓
Digital image array
```

Two concepts are fundamental:

- **Sampling** decides *where* measurements are taken in space.
- **Quantization** decides *which numerical values* can represent the measured intensity.

### 2.1 Sampling

Suppose a continuous image is written as:

$$
f(x,y)
$$

A digital sensor measures it only at discrete spatial positions:

$$
f[m,n]
$$

where `m` and `n` are integer indices.

More samples generally allow finer spatial detail to be represented.

In [ ]:
# Build a smooth synthetic image so spatial sampling can be demonstrated clearly.
x = np.linspace(0, 2 * np.pi, 256)
y = np.linspace(0, 2 * np.pi, 256)
xx, yy = np.meshgrid(x, y)

continuous_like = (
    0.55
    + 0.25 * np.sin(2.0 * xx)
    + 0.20 * np.cos(3.0 * yy)
)
continuous_like = np.clip(continuous_like, 0.0, 1.0)

# Simulate coarser spatial sampling by keeping fewer grid locations.
sampling_steps = [1, 4, 8, 16]

fig, axes = plt.subplots(1, 4, figsize=(14, 3.4))

for ax, step in zip(axes, sampling_steps):
    sampled = continuous_like[::step, ::step]
    ax.imshow(sampled, cmap="gray", vmin=0, vmax=1, interpolation="nearest")
    ax.set_title(f"Sampling step = {step}\nshape = {sampled.shape}")
    ax.axis("off")

fig.suptitle("Spatial Sampling")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "01_sampling.png", dpi=300, bbox_inches="tight")
plt.show()

### Interpretation

As the sampling step increases:

- the number of spatial samples decreases;
- fine detail becomes harder to represent;
- the pixel grid becomes more visible.

**Sampling changes spatial resolution.**

### Common pitfall

Increasing the displayed size of a low-resolution image does **not** recreate missing spatial information. It only enlarges the existing samples.

### 2.2 Quantization

After spatial sampling, measured intensities must also be represented with a finite number of levels.

For a bit depth of $b$:

$$
L = 2^b
$$

where $L$ is the number of representable intensity levels.

Examples:

- 1 bit → 2 levels;
- 2 bits → 4 levels;
- 4 bits → 16 levels;
- 8 bits → 256 levels.

In [ ]:
gradient = np.tile(np.linspace(0, 1, 512), (90, 1))

bit_depths = [1, 2, 4, 8]

fig, axes = plt.subplots(4, 1, figsize=(11, 6))

for ax, bits in zip(axes, bit_depths):
    levels = 2 ** bits

    # Quantize [0, 1] values to a finite set of intensity levels.
    quantized = np.round(gradient * (levels - 1)) / (levels - 1)

    ax.imshow(quantized, cmap="gray", vmin=0, vmax=1, aspect="auto")
    ax.set_title(f"{bits}-bit quantization → {levels} intensity levels")
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "02_quantization.png", dpi=300, bbox_inches="tight")
plt.show()

### Interpretation

Low bit depth creates visible intensity bands because only a small number of gray levels are available.

**Quantization changes intensity resolution**, not spatial resolution.

## 3. Pixels, Coordinates, and Image Matrices

A **pixel** is one spatial sample of a digital image.

A grayscale image can be represented as a 2-D array:

$$
I[y,x]
$$

where:

- `y` = row index;
- `x` = column index;
- the stored number is the pixel intensity.

In image-processing mathematics we often write $I(x,y)$, but NumPy uses:

```python
image[row, column]
image[y, x]
```

This distinction is extremely important.

In [ ]:
toy_gray = np.array(
    [
        [0, 32, 64, 96, 128],
        [24, 56, 88, 120, 152],
        [48, 80, 112, 144, 176],
        [72, 104, 136, 168, 208],
        [96, 128, 160, 208, 255],
    ],
    dtype=np.uint8,
)

fig, ax = plt.subplots(figsize=(5, 4.5))
ax.imshow(toy_gray, cmap="gray", vmin=0, vmax=255)

for row in range(toy_gray.shape[0]):
    for col in range(toy_gray.shape[1]):
        ax.text(
            col,
            row,
            str(toy_gray[row, col]),
            ha="center",
            va="center",
            fontsize=8,
        )

ax.set_title("A grayscale image is a matrix of intensities")
ax.set_xlabel("x / column")
ax.set_ylabel("y / row")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "03_grayscale_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

print("Shape:", toy_gray.shape)
print("dtype:", toy_gray.dtype)
print("Pixel at row=2, column=3:", toy_gray[2, 3])

### Coordinate convention

Most image libraries place the origin at the **top-left** corner:

```text
(0,0) ─────────────→ x / columns
  │
  │
  │
  ↓
 y / rows
```

Therefore:

```python
image[y, x]
```

is the safe mental model.

### Common pitfall

The tuple `(height, width)` returned by `image.shape` is **not** `(width, height)`.

For an RGB image:

```python
height, width, channels = image.shape
```

## 4. Binary, Grayscale, and RGB Images

Three basic image types appear constantly in image processing.

### Binary image

Contains two logical states, often:

- 0 = background;
- 1 or 255 = foreground.

### Grayscale image

Contains one intensity value per pixel.

Typical shape:

```text
(H, W)
```

### RGB image

Contains three values per pixel:

```text
R = red
G = green
B = blue
```

Typical shape:

```text
(H, W, 3)
```

In [ ]:
binary = np.zeros((120, 160), dtype=np.uint8)
binary[30:90, 45:120] = 255

grayscale = np.tile(
    np.linspace(0, 255, 160, dtype=np.uint8),
    (120, 1),
)

rgb = np.zeros((120, 160, 3), dtype=np.uint8)
rgb[:, :53] = [255, 0, 0]
rgb[:, 53:106] = [0, 255, 0]
rgb[:, 106:] = [0, 0, 255]

fig, axes = plt.subplots(1, 3, figsize=(11, 3.3))

axes[0].imshow(binary, cmap="gray", vmin=0, vmax=255)
axes[0].set_title(f"Binary\nshape={binary.shape}")

axes[1].imshow(grayscale, cmap="gray", vmin=0, vmax=255)
axes[1].set_title(f"Grayscale\nshape={grayscale.shape}")

axes[2].imshow(rgb)
axes[2].set_title(f"RGB\nshape={rgb.shape}")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "04_image_types.png", dpi=300, bbox_inches="tight")
plt.show()

### Interpretation

The array dimensionality tells us a lot about the image:

- 2-D does **not automatically mean grayscale**; it could also be a binary mask or label map.
- 3-D does **not automatically mean RGB**; the last dimension must be interpreted according to the image format or library convention.

## 5. Load and Inspect Real Images

Before processing an unfamiliar image, inspect:

1. shape;
2. dtype;
3. minimum and maximum values;
4. number of channels;
5. visual appearance.

This avoids many silent errors.

In [ ]:
images = {}

for name, path in IMAGE_FILES.items():
    # Convert all reference images to RGB so the examples use one consistent format.
    images[name] = np.asarray(Image.open(path).convert("RGB"))

for name, image in images.items():
    print(
        f"{name:9s} | "
        f"shape={str(image.shape):16s} "
        f"dtype={image.dtype} "
        f"range=[{image.min()}, {image.max()}]"
    )

In [ ]:
fig, axes = plt.subplots(1, len(images), figsize=(16, 4))

for ax, (name, image) in zip(axes, images.items()):
    ax.imshow(image)
    ax.set_title(f"{name}\n{image.shape[1]}×{image.shape[0]}")
    ax.axis("off")

fig.suptitle("Reference Images")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "05_reference_images.png", dpi=300, bbox_inches="tight")
plt.show()

### Interpretation

Different images can have different:

- spatial dimensions;
- aspect ratios;
- intensity distributions;
- texture;
- color content.

A robust pipeline should inspect these properties instead of assuming them.

## 6. Dimensions, Resolution, Aspect Ratio, and Channels

For an RGB array with:

```python
image.shape == (H, W, 3)
```

- `H` = height in pixels;
- `W` = width in pixels;
- `3` = number of channels.

The total number of pixels is:

$$
N = H \times W
$$

The aspect ratio is commonly:

$$
\text{aspect ratio} = \frac{W}{H}
$$

In [ ]:
peppers = images["peppers"]

height, width, channels = peppers.shape
pixel_count = height * width
aspect_ratio = width / height

print(f"Height       : {height} pixels")
print(f"Width        : {width} pixels")
print(f"Channels     : {channels}")
print(f"Pixel count  : {pixel_count:,}")
print(f"Aspect ratio : {aspect_ratio:.3f}")

### Pixel dimensions vs physical resolution

A `1920 × 1080` image describes the **number of pixels**, not the physical size of an object.

Physical spatial resolution may require acquisition metadata such as:

- pixel spacing;
- sensor size;
- field of view;
- microscope magnification;
- camera calibration.

Array dimensions alone are not enough.

## 7. Data Types, Bit Depth, Dynamic Range, and Memory

A NumPy image has a data type (`dtype`).

Typical examples:

| dtype | Typical use | Example range |
|---|---|---|
| `bool` | binary logic | `False`, `True` |
| `uint8` | standard images | 0–255 |
| `uint16` | higher bit-depth imaging | 0–65535 |
| `float32` | processing / ML | often 0–1 |
| `float64` | numerical calculations | application-dependent |

For an unsigned 8-bit integer:

$$
0 \le I \le 255
$$

because:

$$
2^8 = 256
$$

possible values are available.

In [ ]:
print("dtype:", peppers.dtype)
print("bytes per value:", peppers.dtype.itemsize)
print("array memory:", f"{peppers.nbytes:,} bytes")
print("array memory:", f"{peppers.nbytes / 1024**2:.3f} MiB")

# NumPy can tell us the legal range of uint8 directly.
uint8_info = np.iinfo(np.uint8)
print("uint8 range:", uint8_info.min, "to", uint8_info.max)

### Dynamic range

The **available range** of `uint8` is 0–255.

The **used dynamic range** of a particular image is:

$$
I_{\max} - I_{\min}
$$

An image can be stored in `uint8` but use only a small part of 0–255, which often produces low contrast.

## 8. Display Images Correctly

Displaying an array is not always trivial.

For grayscale images, Matplotlib may automatically rescale values unless `vmin` and `vmax` are specified.

For fair visual comparison, explicit display limits are often necessary.

In [ ]:
low_contrast = np.linspace(90, 165, 256, dtype=np.uint8)
low_contrast = np.tile(low_contrast, (120, 1))

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))

axes[0].imshow(low_contrast, cmap="gray")
axes[0].set_title("Automatic display scaling")

axes[1].imshow(low_contrast, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Fixed display range: 0–255")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "06_display_scaling.png", dpi=300, bbox_inches="tight")
plt.show()

### Common pitfall

Two images may appear equally contrasted if Matplotlib independently rescales them.

For scientific comparison, always ask whether the display system is using the same intensity limits.

## 9. Pixel Access and Safe Modification

A color pixel is a vector:

```python
[R, G, B]
```

To inspect one pixel:

```python
pixel = image[y, x]
```

To modify an image while keeping the original unchanged, use:

```python
copy = image.copy()
```

In [ ]:
ballons = images["ballons"]

y, x = 140, 220
original_pixel = ballons[y, x].copy()

edited_ballons = ballons.copy()

# Replace a small square around the selected pixel with magenta.
radius = 6
edited_ballons[
    y - radius : y + radius + 1,
    x - radius : x + radius + 1,
] = [255, 0, 255]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].imshow(ballons)
axes[0].scatter([x], [y], s=70, facecolors="none", edgecolors="yellow")
axes[0].set_title("Original + selected pixel")

axes[1].imshow(edited_ballons)
axes[1].set_title("Edited copy")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "07_pixel_edit.png", dpi=300, bbox_inches="tight")
plt.show()

print("Selected coordinate (x, y):", (x, y))
print("Stored RGB value:", original_pixel)

### Interpretation

Direct pixel editing is useful for:

- learning;
- debugging;
- annotations;
- small synthetic experiments.

For large-scale processing, vectorized NumPy operations are usually better than Python loops.

## 10. Regions of Interest (ROI)

A **region of interest** is a selected subregion of an image.

With NumPy slicing:

```python
roi = image[y_start:y_end, x_start:x_end]
```

The vertical range comes first because arrays are indexed as:

```python
[row, column]
```

In [ ]:
tower = images["tower"]

y0, y1 = 120, 360
x0, x1 = 170, 390

roi = tower[y0:y1, x0:x1]

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

axes[0].imshow(tower)
axes[0].add_patch(
    plt.Rectangle(
        (x0, y0),
        x1 - x0,
        y1 - y0,
        fill=False,
        edgecolor="red",
        linewidth=2,
    )
)
axes[0].set_title("Full image and ROI")

axes[1].imshow(roi)
axes[1].set_title(f"ROI: {roi.shape[1]}×{roi.shape[0]}")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "08_region_of_interest.png", dpi=300, bbox_inches="tight")
plt.show()

### Why ROIs matter

ROIs allow us to:

- focus computation on the relevant part of an image;
- reduce processing cost;
- compute local statistics;
- analyze an object independently of the full scene.

## 11. Pixel Neighborhoods

Many image-processing operations depend on a pixel **and its neighbors**.

A common $3\times3$ neighborhood around pixel $(x,y)$ is:

$$
\begin{bmatrix}
I(y-1,x-1) & I(y-1,x) & I(y-1,x+1) \\
I(y,x-1)   & I(y,x)   & I(y,x+1)   \\
I(y+1,x-1) & I(y+1,x) & I(y+1,x+1)
\end{bmatrix}
$$

This idea becomes essential in the spatial-filtering lab.

In [ ]:
einstein_rgb = images["einstein"]

# Convert once to grayscale for easier neighborhood interpretation.
einstein_gray_float = (
    0.299 * einstein_rgb[..., 0].astype(np.float32)
    + 0.587 * einstein_rgb[..., 1].astype(np.float32)
    + 0.114 * einstein_rgb[..., 2].astype(np.float32)
)
einstein_gray = np.clip(einstein_gray_float, 0, 255).astype(np.uint8)

y, x = 120, 120
patch_3x3 = einstein_gray[y - 1 : y + 2, x - 1 : x + 2]

print("Center pixel:", einstein_gray[y, x])
print("3×3 neighborhood:")
print(patch_3x3)

### Border issue

A neighborhood around a border pixel may extend outside the array.

Later filtering operations must define a border strategy such as:

- zero padding;
- reflection;
- replication;
- wrap-around.

Never ignore boundary behavior.

## 12. RGB Channel Decomposition

An RGB image can be interpreted as three separate 2-D planes:

$$
I_R(y,x), \qquad I_G(y,x), \qquad I_B(y,x)
$$

In NumPy:

```python
red   = image[..., 0]
green = image[..., 1]
blue  = image[..., 2]
```

In [ ]:
red = peppers[..., 0]
green = peppers[..., 1]
blue = peppers[..., 2]

fig, axes = plt.subplots(1, 4, figsize=(14, 4))

axes[0].imshow(peppers)
axes[0].set_title("RGB")

axes[1].imshow(red, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Red channel")

axes[2].imshow(green, cmap="gray", vmin=0, vmax=255)
axes[2].set_title("Green channel")

axes[3].imshow(blue, cmap="gray", vmin=0, vmax=255)
axes[3].set_title("Blue channel")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "09_rgb_channels.png", dpi=300, bbox_inches="tight")
plt.show()

print(
    "Channel means:",
    {
        "R": round(float(red.mean()), 2),
        "G": round(float(green.mean()), 2),
        "B": round(float(blue.mean()), 2),
    },
)

### Interpretation

A bright region in one channel means that color component has a high value there.

Channel decomposition is important for:

- color analysis;
- segmentation;
- color correction;
- feature extraction.

## 13. RGB and BGR Conventions

Different libraries may use different channel orders.

- **Pillow** → RGB
- **Matplotlib** → expects RGB
- **OpenCV** → traditionally loads color images as BGR

If BGR data is displayed as RGB, red and blue are exchanged.

In [ ]:
rgb_example = peppers
bgr_like = rgb_example[..., ::-1]

fig, axes = plt.subplots(1, 2, figsize=(9, 4))

axes[0].imshow(rgb_example)
axes[0].set_title("Correct RGB")

axes[1].imshow(bgr_like)
axes[1].set_title("Channels reversed")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "10_rgb_bgr.png", dpi=300, bbox_inches="tight")
plt.show()

### Common pitfall

If skin, sky, or red objects suddenly appear blue, check the channel order before assuming the image data are corrupted.

## 14. RGB-to-Grayscale Conversion

A simple arithmetic mean:

$$
Y = \frac{R+G+B}{3}
$$

treats all channels equally.

A common luminance approximation gives different weights:

$$
Y = 0.299R + 0.587G + 0.114B
$$

because human vision is more sensitive to green than to blue.

In [ ]:
def rgb_to_grayscale(rgb_image: np.ndarray) -> np.ndarray:
    """Convert an RGB uint8 image to grayscale using luminance weights."""

    # Convert before multiplication so arithmetic is performed safely.
    rgb_float = rgb_image.astype(np.float32)

    gray = (
        0.299 * rgb_float[..., 0]
        + 0.587 * rgb_float[..., 1]
        + 0.114 * rgb_float[..., 2]
    )

    # Return a standard 8-bit grayscale image.
    return np.clip(gray, 0, 255).astype(np.uint8)


einstein_rgb = images["einstein"]
einstein_gray = rgb_to_grayscale(einstein_rgb)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))

axes[0].imshow(einstein_rgb)
axes[0].set_title(f"RGB shape: {einstein_rgb.shape}")

axes[1].imshow(einstein_gray, cmap="gray", vmin=0, vmax=255)
axes[1].set_title(f"Grayscale shape: {einstein_gray.shape}")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "11_rgb_to_grayscale.png", dpi=300, bbox_inches="tight")
plt.show()

### What is lost?

Grayscale conversion removes chromatic information.

Two different colors can map to similar grayscale intensities, so grayscale should be used only when color information is not essential to the task.

## 15. Image Statistics

Useful scalar summaries include:

- minimum;
- maximum;
- mean;
- median;
- standard deviation;
- percentiles.

These statistics help characterize global brightness and intensity spread.

In [ ]:
grass_gray = rgb_to_grayscale(images["grass"])

statistics = {
    "min": int(grass_gray.min()),
    "max": int(grass_gray.max()),
    "mean": float(grass_gray.mean()),
    "median": float(np.median(grass_gray)),
    "std": float(grass_gray.std()),
    "p05": float(np.percentile(grass_gray, 5)),
    "p95": float(np.percentile(grass_gray, 95)),
}

for name, value in statistics.items():
    print(f"{name:>6s}: {value:.3f}" if isinstance(value, float) else f"{name:>6s}: {value}")

### Limitation of global statistics

Statistics do **not** describe where pixel values occur.

Two images can have the same mean and histogram while having completely different spatial structures.

## 16. Intensity Histograms

A grayscale histogram counts how many pixels belong to each intensity bin.

For an 8-bit image, a natural choice is 256 bins representing values 0–255.

A histogram can indicate whether an image is:

- globally dark;
- globally bright;
- low contrast;
- spread over a wide dynamic range.

In [ ]:
ballons_gray = rgb_to_grayscale(images["ballons"])

counts, bin_edges = np.histogram(
    ballons_gray.ravel(),
    bins=256,
    range=(0, 256),
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].imshow(ballons_gray, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Grayscale image")
axes[0].axis("off")

axes[1].plot(np.arange(256), counts)
axes[1].set_title("Intensity histogram")
axes[1].set_xlabel("Intensity")
axes[1].set_ylabel("Pixel count")
axes[1].set_xlim(0, 255)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "12_intensity_histogram.png", dpi=300, bbox_inches="tight")
plt.show()

print("Histogram count:", counts.sum())
print("Number of image pixels:", ballons_gray.size)

### Histogram limitation: spatial information is lost

A histogram tells us **how many** pixels have each intensity, but not **where** those pixels are located.

If an image is randomly shuffled, its histogram remains unchanged even though its visual structure is destroyed.

In [ ]:
shuffled = ballons_gray.ravel().copy()
RNG.shuffle(shuffled)
shuffled = shuffled.reshape(ballons_gray.shape)

hist_original, _ = np.histogram(ballons_gray.ravel(), bins=256, range=(0, 256))
hist_shuffled, _ = np.histogram(shuffled.ravel(), bins=256, range=(0, 256))

print("Histograms identical:", np.array_equal(hist_original, hist_shuffled))

fig, axes = plt.subplots(1, 2, figsize=(9, 4))

axes[0].imshow(ballons_gray, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Original")

axes[1].imshow(shuffled, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Pixels shuffled")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

## 17. Dynamic Range and Min-Max Normalization

If an image uses only a narrow intensity interval, contrast may be weak.

Min-max normalization maps:

$$
I_{\min} \rightarrow 0
$$

and:

$$
I_{\max} \rightarrow 255
$$

using:

$$
I_{\mathrm{norm}}
=
\frac{I-I_{\min}}{I_{\max}-I_{\min}}
\times 255
$$

In [ ]:
def minmax_normalize(gray_image: np.ndarray) -> np.ndarray:
    """Stretch a grayscale image to the full uint8 range [0, 255]."""

    image_float = gray_image.astype(np.float32)

    minimum = image_float.min()
    maximum = image_float.max()

    # Avoid division by zero for constant images.
    if maximum == minimum:
        return np.zeros_like(gray_image)

    normalized = (image_float - minimum) / (maximum - minimum)
    normalized *= 255.0

    return np.clip(normalized, 0, 255).astype(np.uint8)


source_gray = rgb_to_grayscale(images["ballons"])

# Artificially compress the intensity range to simulate a low-contrast image.
low_contrast = 90 + (source_gray.astype(np.float32) / 255.0) * 76
low_contrast = np.clip(low_contrast, 0, 255).astype(np.uint8)

normalized = minmax_normalize(low_contrast)

fig, axes = plt.subplots(2, 2, figsize=(10, 7))

axes[0, 0].imshow(low_contrast, cmap="gray", vmin=0, vmax=255)
axes[0, 0].set_title("Low-contrast image")
axes[0, 0].axis("off")

axes[0, 1].hist(low_contrast.ravel(), bins=256, range=(0, 256))
axes[0, 1].set_title("Before normalization")
axes[0, 1].set_xlabel("Intensity")

axes[1, 0].imshow(normalized, cmap="gray", vmin=0, vmax=255)
axes[1, 0].set_title("After min-max normalization")
axes[1, 0].axis("off")

axes[1, 1].hist(normalized.ravel(), bins=256, range=(0, 256))
axes[1, 1].set_title("After normalization")
axes[1, 1].set_xlabel("Intensity")

fig.tight_layout()
fig.savefig(
    OUTPUT_DIR / "13_dynamic_range_normalization.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

print("Before range:", int(low_contrast.min()), "to", int(low_contrast.max()))
print("After range :", int(normalized.min()), "to", int(normalized.max()))

### Important distinction

Min-max normalization stretches the observed range.

It is **not** the same as histogram equalization, which changes the intensity distribution using the cumulative histogram.

## 18. `uint8` Arithmetic, Overflow, Clipping, and Floating Point

This is one of the most important practical topics for beginners.

`uint8` can store only 0–255. Values outside this range cannot be represented.

For safe arithmetic:

1. convert to a wider or floating-point type;
2. perform the operation;
3. clip to the valid output range;
4. convert back if necessary.

In [ ]:
value = np.array([250], dtype=np.uint8)

# Unsafe uint8 addition can wrap around.
unsafe = value + np.array([20], dtype=np.uint8)

# Safe workflow.
safe_float = value.astype(np.float32) + 20.0
safe_uint8 = np.clip(safe_float, 0, 255).astype(np.uint8)

print("Original value :", value[0])
print("Unsafe result  :", unsafe[0])
print("Safe result    :", safe_uint8[0])

### Common pitfall

Never assume that arithmetic on `uint8` behaves like ordinary mathematical integers.

Convert before operations that can exceed the legal range.

## 19. Image Noise Fundamentals

Noise is unwanted variation in measured pixel values.

Sources include:

- sensor electronics;
- low-light acquisition;
- photon statistics;
- transmission;
- compression;
- environmental interference.

Common noise models include:

### Gaussian noise
Additive continuous fluctuations.

### Salt-and-pepper noise
Random isolated dark and bright pixels.

### Poisson noise
Signal-dependent counting noise.

### Speckle noise
Multiplicative granular noise.

In [ ]:
base_gray = einstein_gray

# 1) Gaussian noise
gaussian_noise = RNG.normal(0.0, 20.0, size=base_gray.shape)
gaussian = np.clip(
    base_gray.astype(np.float32) + gaussian_noise,
    0,
    255,
).astype(np.uint8)

# 2) Salt-and-pepper noise
salt_pepper = base_gray.copy()
probability = 0.03
random_map = RNG.random(base_gray.shape)
salt_pepper[random_map < probability / 2] = 0
salt_pepper[random_map > 1 - probability / 2] = 255

# 3) Poisson noise
scaled = base_gray.astype(np.float32) / 255.0
poisson = RNG.poisson(scaled * 30.0) / 30.0
poisson = np.clip(poisson * 255.0, 0, 255).astype(np.uint8)

# 4) Speckle noise
speckle_noise = RNG.normal(0.0, 0.18, size=base_gray.shape)
speckle = base_gray.astype(np.float32) * (1.0 + speckle_noise)
speckle = np.clip(speckle, 0, 255).astype(np.uint8)

fig, axes = plt.subplots(1, 5, figsize=(16, 3.6))

examples = [
    ("Original", base_gray),
    ("Gaussian", gaussian),
    ("Salt & pepper", salt_pepper),
    ("Poisson", poisson),
    ("Speckle", speckle),
]

for ax, (title, image) in zip(axes, examples):
    ax.imshow(image, cmap="gray", vmin=0, vmax=255)
    ax.set_title(title)
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "14_noise_models.png", dpi=300, bbox_inches="tight")
plt.show()

### Why noise models matter

Different filters are effective against different types of noise.

For example, a median filter is particularly useful for salt-and-pepper noise, while Gaussian smoothing is often associated with Gaussian-like noise.

Those methods are studied in the spatial-filtering lab.

## 20. Image Comparison Metrics

Suppose $R$ is a reference image and $T$ is a test image.

### Mean Absolute Error (MAE)

$$
\mathrm{MAE}
=
\frac{1}{N}
\sum |R-T|
$$

### Mean Squared Error (MSE)

$$
\mathrm{MSE}
=
\frac{1}{N}
\sum (R-T)^2
$$

### Root Mean Squared Error (RMSE)

$$
\mathrm{RMSE}
=
\sqrt{\mathrm{MSE}}
$$

### Peak Signal-to-Noise Ratio (PSNR)

For 8-bit images:

$$
\mathrm{PSNR}
=
10\log_{10}
\left(
\frac{255^2}{\mathrm{MSE}}
\right)
$$

Higher PSNR usually means greater numerical similarity to the reference.

In [ ]:
def image_metrics(reference: np.ndarray, test: np.ndarray) -> dict:
    """Compute MAE, MSE, RMSE, and PSNR for same-shaped images."""

    if reference.shape != test.shape:
        raise ValueError("Images must have identical shapes.")

    reference_f = reference.astype(np.float64)
    test_f = test.astype(np.float64)

    difference = reference_f - test_f

    mae = np.mean(np.abs(difference))
    mse = np.mean(difference ** 2)
    rmse = np.sqrt(mse)

    if mse == 0:
        psnr = np.inf
    else:
        psnr = 10.0 * np.log10((255.0 ** 2) / mse)

    return {
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "PSNR": psnr,
    }


noise_results = {
    "Gaussian": image_metrics(base_gray, gaussian),
    "Salt & pepper": image_metrics(base_gray, salt_pepper),
    "Poisson": image_metrics(base_gray, poisson),
    "Speckle": image_metrics(base_gray, speckle),
}

for noise_name, metrics in noise_results.items():
    print(noise_name)
    for metric_name, value in metrics.items():
        print(f"  {metric_name:>4s}: {value:.4f}")

### Interpretation

These metrics are objective numerical comparisons, but they do not perfectly model human perception.

A visually important structural error can sometimes produce a moderate MSE, while a visually small global change can affect many pixels.

Use numerical metrics **together with visual inspection**.

## 21. PNG vs JPEG and Image Saving

### PNG

- lossless compression;
- preserves pixel values;
- appropriate for masks, diagrams, labels, and quantitative intermediate results.

### JPEG

- lossy compression;
- efficient for natural photographs;
- decoded values may differ from the original;
- repeated save/load cycles may introduce artifacts.

For quantitative processing, prefer lossless formats unless compression is part of the experiment.

In [ ]:
example = images["peppers"]

png_path = OUTPUT_DIR / "15_saved_example.png"
jpg_path = OUTPUT_DIR / "15_saved_example.jpg"

Image.fromarray(example).save(png_path)
Image.fromarray(example).save(jpg_path, quality=75)

png_reload = np.asarray(Image.open(png_path).convert("RGB"))
jpg_reload = np.asarray(Image.open(jpg_path).convert("RGB"))

png_metrics = image_metrics(example, png_reload)
jpg_metrics = image_metrics(example, jpg_reload)

print("PNG reload MSE :", png_metrics["MSE"])
print("JPEG reload MSE:", jpg_metrics["MSE"])
print("PNG path :", png_path)
print("JPEG path:", jpg_path)

### Expected result

The PNG reload should preserve the pixel values exactly in normal use, producing MSE = 0.

JPEG generally produces a non-zero MSE because the compression is lossy.

## 22. Standard Image Inspection Workflow

Whenever you receive an unfamiliar image, use this sequence:

```text
1. Locate the file
2. Load it
3. Inspect shape
4. Inspect dtype
5. Inspect min / max
6. Determine color/channel convention
7. Display it correctly
8. Compute useful statistics
9. Decide whether conversion is needed
10. Process in a safe numeric dtype
11. Validate the result numerically
12. Validate the result visually
13. Save outputs reproducibly
```

This simple workflow prevents many silent bugs.

In [ ]:
def inspect_image(name: str, image: np.ndarray) -> None:
    """Print a compact summary for an unfamiliar image array."""

    print(f"Name       : {name}")
    print(f"Shape      : {image.shape}")
    print(f"Dimensions : {image.ndim}")
    print(f"dtype      : {image.dtype}")
    print(f"Min / max  : {image.min()} / {image.max()}")
    print(f"Mean       : {image.mean():.3f}")
    print(f"Memory     : {image.nbytes:,} bytes")


inspect_image("peppers", peppers)

## 23. Validation Checks

A good notebook should verify important assumptions explicitly rather than relying only on visual inspection.

In [ ]:
# Shapes
assert peppers.ndim == 3
assert peppers.shape[2] == 3
assert einstein_gray.ndim == 2

# Data types
assert peppers.dtype == np.uint8
assert einstein_gray.dtype == np.uint8

# Histogram consistency
assert counts.sum() == ballons_gray.size

# Normalization range
assert normalized.min() == 0
assert normalized.max() == 255

# Metric self-consistency
self_metrics = image_metrics(einstein_gray, einstein_gray)
assert self_metrics["MAE"] == 0
assert self_metrics["MSE"] == 0
assert self_metrics["RMSE"] == 0
assert np.isinf(self_metrics["PSNR"])

# ROI sanity
assert roi.shape[0] == (y1 - y0)
assert roi.shape[1] == (x1 - x0)

print("All fundamental validation checks passed.")

## Final Result Summary

The notebook implements the complete Image Processing Fundamental workflow and ends with explicit numerical/output validation. Generated visual evidence is stored under `../outputs/figures/`.
